# PrediRuta: construcción de los datasets finales

Este notebook parte de los **seis CSV relacionales depurados**. 
1. construye un diccionario reproducible para convertir las variables categóricas en códigos numéricos
2. construye una cuadrícula recurrente zona-mes-día-hora para modelar la ocurrencia histórica
3. incorpora una climatología consistente a todas las combinaciones de la cuadrícula
4. integra las tablas necesarias para los análisis posteriores

## 0. Configuración

In [ ]:
# Librerías y parámetros generales
from pathlib import Path
from itertools import product
import warnings
import numpy as np
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree
from sklearn.cluster import MiniBatchKMeans

In [ ]:
# Configuración de rutas y carpetas
CARPETA_TABLAS = Path('data_procesada') / 'tablas_finales'
CARPETA_CACHE_CLIMA = Path('data_procesada') / 'cache_clima_hibrido'
CARPETA_SALIDA = Path('data_procesada') / 'modelado'

# Crear carpeta de salida si no existe
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

# Rutas de salida de archivos
RUTA_DICCIONARIO = CARPETA_SALIDA / 'DICCIONARIO_CATEGORIAS.csv'
RUTA_OCURRENCIA = CARPETA_SALIDA / 'DATASET_OCURRENCIA.csv'
RUTA_COMPLETA = CARPETA_SALIDA / 'TABLA_COMPLETA_ACCIDENTES.csv'

## 1. Carga de los CSV separados

Los archivos originales no se sobrescriben. Todas las transformaciones se aplican sobre copias en memoria.

In [ ]:
# Archivos de entrada
ARCHIVOS = {
    'ACCIDENTE': 'ACCIDENTE.csv',
    'CLIMA': 'CLIMA.csv',
    'VIA': 'VIA.csv',
    'VEHICULO': 'VEHICULO.csv',
    'ACTOR_VIAL': 'ACTOR_VIAL.csv',
    'CAUSA': 'CAUSA.csv',
}

print('Archivos encontrados:', len(ARCHIVOS))

In [ ]:
# Cargar las seis tablas
TABLAS = {}

for tabla, archivo in ARCHIVOS.items():

    ruta = CARPETA_TABLAS / archivo

    if tabla == 'ACCIDENTE':
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False,
            parse_dates=['FECHA_HORA']
        )

    elif tabla == 'CLIMA':
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False,
            parse_dates=['FECHA_HORA_CLIMA']
        )

    else:
        TABLAS[tabla] = pd.read_csv(
            ruta,
            encoding='utf-8-sig',
            low_memory=False
        )

# Resumen de tablas cargadas
inventario = pd.DataFrame([
    {
        'TABLA': tabla,
        'FILAS': len(datos),
        'COLUMNAS': len(datos.columns),
        'ARCHIVO': ARCHIVOS[tabla]
    }
    for tabla, datos in TABLAS.items()
])

display(inventario)

## 2. Diccionario de variables categóricas

En esta sección se identifican las variables categóricas presentes en cada una de las tablas y se construye un diccionario que permita convertir sus valores a códigos numéricos.

Cada combinación de tabla y columna cuenta con su propio catálogo de categorías, siguiendo estas reglas:

- El código `0` representa los valores `SIN INFORMACION`.
- Las demás categorías se organizan alfabéticamente.
- La codificación comienza en `1` y continúa de forma consecutiva.

Para la tabla de causas se valida la relación entre `CODIGO_CAUSA` y `NOMBRE`. Para el modelado se utiliza `NOMBRE`, porque contiene el significado de la causa. `CODIGO_CAUSA` se usa solamente para revisar la consistencia de los datos y luego se excluye de la codificación.

El diccionario generado se conserva como referencia para interpretar posteriormente los códigos asignados y garantizar que la misma transformación pueda aplicarse de manera consistente en las siguientes etapas del análisis.

In [ ]:
# Validación de consistencia entre códigos y nombres en la tabla CAUSA
causas_catalogo = TABLAS['CAUSA'][['CODIGO_CAUSA', 'NOMBRE']].copy()

for columna in ['CODIGO_CAUSA', 'NOMBRE']:
    causas_catalogo[columna] = (
        causas_catalogo[columna]
        .astype('string')
        .str.strip()
        .replace('', pd.NA)
    )

# Validación de consistencia entre códigos y nombres
pares_causa = causas_catalogo.dropna().drop_duplicates()
nombres_por_codigo = pares_causa.groupby('CODIGO_CAUSA')['NOMBRE'].nunique()
codigos_por_nombre = pares_causa.groupby('NOMBRE')['CODIGO_CAUSA'].nunique()
codigos_inconsistentes = nombres_por_codigo[nombres_por_codigo > 1]

# Validación de consistencia entre las dos columnas
validacion_causas = pd.DataFrame({
    'METRICA': [
        'Códigos distintos',
        'Nombres distintos',
        'Códigos con varios nombres',
        'Nombres asociados a varios códigos'
    ],
    'VALOR': [
        causas_catalogo['CODIGO_CAUSA'].nunique(),
        causas_catalogo['NOMBRE'].nunique(),
        int((nombres_por_codigo > 1).sum()),
        int((codigos_por_nombre > 1).sum())
    ],
})

display(validacion_causas)

In [ ]:
# Listado de nombres que agrupan más de un código administrativo
nombres_con_varios_codigos = (
    codigos_por_nombre[codigos_por_nombre > 1]
    .rename('CANTIDAD_CODIGOS')
    .reset_index()
    .sort_values('CANTIDAD_CODIGOS', ascending=False)
)

display(nombres_con_varios_codigos)

Cada `CODIGO_CAUSA` está asociado a un único valor de `NOMBRE`, por lo que no se identifican inconsistencias entre ambas columnas. Sin embargo, se observa que algunos nombres genéricos pueden estar relacionados con varios códigos administrativos.

Para el modelo se conserva la variable `NOMBRE`, ya que representa de forma directa el significado de la causa y facilita su interpretación. A partir de sus categorías se generan los códigos numéricos definidos en el diccionario.

Por esta razón, `CODIGO_CAUSA` se conserva únicamente como referencia dentro de la información original y no se utiliza como variable predictora en el modelo.

In [ ]:
# Variables categóricas de cada tabla
COLUMNAS_CATEGORICAS = {
    'ACCIDENTE': ['CLASE_ACCIDENTE'],
    'CLIMA': [],
    'VIA': [
        'GEOMETRIA_PLANTA','GEOMETRIA_TERRENO','GEOMETRIA_SECCION','SENTIDO_VIA',
        'SUPERFICIE_RODADURA','ESTADO_VIA','CONDICION_VIA',
        'ILUMINACION_ARTIFICIAL','SEMAFORO',
    ],
    'VEHICULO': ['CLASE','SERVICIO'],
    'ACTOR_VIAL': ['CONDICION','ESTADO','GENERO'],
    'CAUSA': ['NOMBRE', 'TIPO'],
}

In [ ]:
# Construir diccionario de categorías y mapas de transformación
SIN_INFORMACION = 'SIN INFORMACION'

registros_diccionario = []
MAPAS_CATEGORIAS = {}

for tabla, columnas in COLUMNAS_CATEGORICAS.items():
    for columna in columnas:

        # Limpiar categorías
        valores = (
            TABLAS[tabla][columna]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
            .fillna(SIN_INFORMACION)
        )

        # Ordenar categorías alfabéticamente
        categorias = sorted(set(valores) - {SIN_INFORMACION})

        # Crear mapa numérico
        mapa = {
            SIN_INFORMACION: 0,
            **{
                categoria: codigo
                for codigo, categoria in enumerate(categorias, start=1)
            }
        }

        MAPAS_CATEGORIAS[(tabla, columna)] = mapa

        # Agregar categorías al diccionario
        registros_diccionario.extend([
            [tabla, columna, codigo, valor]
            for valor, codigo in mapa.items()
        ])

# Consolidar diccionario
DICCIONARIO_CATEGORIAS = (
    pd.DataFrame(
        registros_diccionario,
        columns=['TABLA', 'COLUMNA', 'CODIGO', 'DESCRIPCION']
    )
    .sort_values(['TABLA', 'COLUMNA', 'CODIGO'])
    .reset_index(drop=True)
)

display(DICCIONARIO_CATEGORIAS)

In [ ]:
# Exportar el diccionario
DICCIONARIO_CATEGORIAS.to_csv(
    RUTA_DICCIONARIO, index=False, encoding='utf-8-sig'
)
print(f'Diccionario exportado: {RUTA_DICCIONARIO}')

## 3. Transformación numérica de las seis tablas

Para cada una de las seis tablas se crea una copia sobre la cual se realiza la transformación de las variables categóricas, reemplazando sus valores originales por los códigos numéricos definidos previamente en el diccionario.

Las tablas originales se conservan sin modificaciones. Las llaves de identificación, las fechas y las variables que ya son numéricas mantienen sus valores originales.

In [ ]:
# Aplicar codificación a las variables categóricas
TABLAS_CODIFICADAS = {tabla: datos.copy() for tabla, datos in TABLAS.items()}

resumen_codificacion = []

for tabla, columnas in COLUMNAS_CATEGORICAS.items():
    for columna in columnas: # Iterar sobre las columnas categóricas de cada tabla
        valores = (
            TABLAS_CODIFICADAS[tabla][columna]
            .astype('string')
            .str.strip()
            .replace('', pd.NA)
            .fillna(SIN_INFORMACION)
        )

        codigos = valores.map(MAPAS_CATEGORIAS[(tabla, columna)])

        TABLAS_CODIFICADAS[tabla][columna] = codigos.astype('int16')

        resumen_codificacion.append([
            tabla,
            columna,
            valores.nunique(),
            codigos.min(),
            codigos.max()
        ])

# Crear un resumen de la codificación aplicada
resumen_codificacion = pd.DataFrame(
    resumen_codificacion,
    columns=[
        'TABLA',
        'COLUMNA',
        'CATEGORIAS',
        'CODIGO_MINIMO',
        'CODIGO_MAXIMO'
    ]
)

display(resumen_codificacion)

In [ ]:
# Exportar tablas codificadas
for tabla, datos in TABLAS_CODIFICADAS.items():
    datos.to_csv(
        CARPETA_SALIDA / f'{tabla}.csv',
        index=False,
        encoding='utf-8-sig'
    )

print(f'Tablas codificadas exportadas en: {CARPETA_SALIDA}')

## 4. Construcción del dataset de ocurrencia por patrones recurrentes

El dataset de ocurrencia se construye únicamente a partir de los accidentes históricos. Primero se consolidan los eventos positivos por celda de 500 metros y hora; después esas observaciones se agrupan en 150 zonas y en patrones recurrentes de mes, día de la semana y hora. Finalmente se genera la cuadrícula completa de patrones y se asigna `OCURRIO_ACCIDENTE = 0` a las combinaciones que no aparecen entre los casos históricos. No se generan controles desplazando fechas ni se conservan pares caso-control.

### 4.1 Casos históricos y celdas espaciales

Los accidentes se normalizan inicialmente como observaciones **celda-hora**. Para esto, Bogotá se divide en celdas regulares de 500 metros utilizando el sistema de coordenadas `EPSG:3116`.

Esta granularidad se adopta como un punto de partida para mantener un equilibrio entre el nivel de detalle espacial y la cantidad de observaciones disponibles para el entrenamiento del modelo. Posteriormente, su tamaño podrá validarse y ajustarse de acuerdo con la distribución de los accidentes y el desempeño obtenido.

Cada accidente se asigna a la celda espacial y a la hora correspondiente. Cuando se presentan varios accidentes dentro de la misma celda y durante la misma hora, estos se agrupan en una sola observación positiva, indicando que en esa combinación de espacio y tiempo ocurrió al menos un accidente.

Estas observaciones positivas se utilizarán para crear 150 zonas mediante `MiniBatchKMeans`. El dataset final tendrá una fila por combinación `ZONA_CIUDAD`, `MES`, `DIA_SEMANA` y `HORA`; no conservará una fecha ni un año específicos.

In [ ]:
# Preparar los casos históricos para el dataset de ocurrencia
TAMANO_CELDA_METROS = 500
NUMERO_ZONAS = 150
SEMILLA = 42

# Construir dataset de ocurrencia de accidentes por celda y hora
accidentes = TABLAS_CODIFICADAS['ACCIDENTE'][
    ['ACCIDENTE_ID', 'FECHA_HORA', 'LATITUD', 'LONGITUD']
].copy()

# Asegurar formato numérico
accidentes[['LATITUD', 'LONGITUD']] = accidentes[
    ['LATITUD', 'LONGITUD']
].apply(pd.to_numeric, errors='coerce')
accidentes = accidentes.dropna(
    subset=['FECHA_HORA', 'LATITUD', 'LONGITUD']
).copy()

# Redondear fecha y hora
accidentes['FECHA_HORA'] = accidentes['FECHA_HORA'].dt.floor('h')

# Transformar coordenadas geográficas a coordenadas planas (EPSG:3116)
wgs84_a_bogota = Transformer.from_crs(
    'EPSG:4326',
    'EPSG:3116',
    always_xy=True
)

bogota_a_wgs84 = Transformer.from_crs(
    'EPSG:3116',
    'EPSG:4326',
    always_xy=True
)

x, y = wgs84_a_bogota.transform(
    accidentes['LONGITUD'].to_numpy(),
    accidentes['LATITUD'].to_numpy()
)

# Asignar cada accidente a una celda de 500 m
accidentes['CELDA_X'] = (
    np.floor(np.asarray(x) / TAMANO_CELDA_METROS)
    .astype('int32')
)

accidentes['CELDA_Y'] = (
    np.floor(np.asarray(y) / TAMANO_CELDA_METROS)
    .astype('int32')
)

accidentes['CELDA_ID'] = (
    accidentes['CELDA_X'].astype(str)
    + '_'
    + accidentes['CELDA_Y'].astype(str)
)

# Construir casos positivos únicos por celda y hora
casos = (
    accidentes[
        ['CELDA_ID', 'CELDA_X', 'CELDA_Y', 'FECHA_HORA']
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Obtener coordenadas del centro de cada celda
x_centro = (
    casos['CELDA_X'].to_numpy() + 0.5
) * TAMANO_CELDA_METROS

y_centro = (
    casos['CELDA_Y'].to_numpy() + 0.5
) * TAMANO_CELDA_METROS

lon_centro, lat_centro = bogota_a_wgs84.transform(
    x_centro,
    y_centro
)

casos['LATITUD'] = lat_centro
casos['LONGITUD'] = lon_centro
# Derivar las categorías temporales usadas por la cuadrícula
casos['MES'] = casos['FECHA_HORA'].dt.month
casos['DIA_SEMANA'] = casos['FECHA_HORA'].dt.dayofweek
casos['HORA'] = casos['FECHA_HORA'].dt.hour
casos['OCURRIO_ACCIDENTE'] = 1

display(casos.head())

print(f'Casos históricos positivos celda-hora: {len(casos):,}')
print(f'Celdas utilizadas: {casos["CELDA_ID"].nunique():,}')

### 4.2 Zonas y combinaciones sin accidente histórico

Las observaciones positivas se agrupan geográficamente en 150 zonas mediante `MiniBatchKMeans`. Después se agregan por `ZONA_CIUDAD`, `MES`, `DIA_SEMANA` y `HORA`. Una combinación recibe `OCURRIO_ACCIDENTE = 1` cuando existe al menos un caso histórico en cualquiera de los años disponibles.

Se construye la cuadrícula completa de `150 × 12 × 7 × 24 = 302.400` combinaciones. Las combinaciones que no aparecen entre los casos históricos se completan con `OCURRIO_ACCIDENTE = 0`.

> **Interpretación:** los ceros no representan controles observados en una fecha específica. Representan patrones recurrentes para los que no se encontró un accidente en el histórico. En consecuencia, la variable objetivo expresa ocurrencia histórica acumulada y no una probabilidad por celda-hora concreta.

In [ ]:
# Agrupar los casos históricos en zonas geográficas
coordenadas = casos[['LATITUD', 'LONGITUD']]

kmeans = MiniBatchKMeans(
    n_clusters=NUMERO_ZONAS,
    random_state=SEMILLA,
    batch_size=10_000,
    n_init=3
)
casos['ZONA_CIUDAD'] = kmeans.fit_predict(coordenadas)

# Conservar los centroides para ubicar cada zona y asignar el clima
centroides_zona = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=['LATITUD_CENTROIDE', 'LONGITUD_CENTROIDE']
).rename_axis('ZONA_CIUDAD').reset_index()
centroides_zona['ZONA_CIUDAD'] = centroides_zona['ZONA_CIUDAD'].astype('int16')

print(f'Zonas construidas: {centroides_zona["ZONA_CIUDAD"].nunique():,}')
display(centroides_zona.head())

In [ ]:
# Consolidar si existió al menos un accidente en cada patrón recurrente
claves_patron = ['ZONA_CIUDAD', 'MES', 'DIA_SEMANA', 'HORA']
patrones_con_accidente = (
    casos.groupby(claves_patron, as_index=False)
    .agg(OCURRIO_ACCIDENTE=('OCURRIO_ACCIDENTE', 'max'))
)

# Construir todas las combinaciones posibles de zona, mes, día y hora
cuadricula = pd.DataFrame(
    product(
        range(NUMERO_ZONAS),
        range(1, 13),
        range(0, 7),
        range(0, 24)
    ),
    columns=claves_patron
)

dataset_ocurrencia = cuadricula.merge(
    patrones_con_accidente,
    on=claves_patron,
    how='left',
    validate='1:1'
)
dataset_ocurrencia['OCURRIO_ACCIDENTE'] = (
    dataset_ocurrencia['OCURRIO_ACCIDENTE']
    .fillna(0)
    .astype('int8')
)
dataset_ocurrencia = dataset_ocurrencia.merge(
    centroides_zona,
    on='ZONA_CIUDAD',
    how='left',
    validate='m:1'
)

print(f'Combinaciones generadas: {len(dataset_ocurrencia):,}')
display(dataset_ocurrencia['OCURRIO_ACCIDENTE'].value_counts().rename('FILAS'))

### 4.3 Climatología de las zonas y patrones recurrentes

La cuadrícula recurrente no contiene un año ni una fecha específicos. Por esta razón, a cada zona y combinación `MES`–`DIA_SEMANA`–`HORA` se le asigna el promedio histórico del clima disponible en el nodo meteorológico más cercano.

Se combinan los periodos cubiertos por `ERA5` y `ECMWF IFS`, ponderando los promedios por la cantidad de observaciones horarias de cada fuente. La dirección del viento se promedia de forma circular.

El clima se calcula para **todas** las filas, tanto para `OCURRIO_ACCIDENTE = 1` como para `OCURRIO_ACCIDENTE = 0`. Esto evita que la ausencia de clima revele directamente la clase objetivo.

In [ ]:
# Construir el catálogo de cachés climáticos
archivos_cache = sorted(CARPETA_CACHE_CLIMA.glob('clima_*.csv'))

catalogo_nodos = []
for ruta in archivos_cache:
    muestra = pd.read_csv(ruta, nrows=1)
    catalogo_nodos.append([
        ruta.name, muestra.loc[0,'MODELO'],
        float(muestra.loc[0,'LATITUD_CELDA']),
        float(muestra.loc[0,'LONGITUD_CELDA']),
    ])
catalogo_nodos = pd.DataFrame(catalogo_nodos, columns=[
    'ARCHIVO_CACHE','MODELO_CLIMA','LATITUD_NODO','LONGITUD_NODO',
]).drop_duplicates('ARCHIVO_CACHE')

display(catalogo_nodos.groupby('MODELO_CLIMA').size().rename('NODOS').reset_index())

In [ ]:
# Asignar a cada zona el nodo más cercano de cada modelo climático
x_zonas, y_zonas = wgs84_a_bogota.transform(
    centroides_zona['LONGITUD_CENTROIDE'].to_numpy(),
    centroides_zona['LATITUD_CENTROIDE'].to_numpy()
)
coordenadas_zonas = np.column_stack([x_zonas, y_zonas])

asignaciones_zona_clima = []
for modelo, nodos in catalogo_nodos.groupby('MODELO_CLIMA'):
    x_nodos, y_nodos = wgs84_a_bogota.transform(
        nodos['LONGITUD_NODO'].to_numpy(),
        nodos['LATITUD_NODO'].to_numpy()
    )
    arbol = cKDTree(np.column_stack([x_nodos, y_nodos]))
    _, posiciones = arbol.query(coordenadas_zonas, k=1)

    asignaciones_zona_clima.append(pd.DataFrame({
        'ZONA_CIUDAD': centroides_zona['ZONA_CIUDAD'].to_numpy(),
        'MODELO_CLIMA': modelo,
        'ARCHIVO_CACHE': nodos.iloc[posiciones]['ARCHIVO_CACHE'].to_numpy()
    }))

asignaciones_zona_clima = pd.concat(
    asignaciones_zona_clima,
    ignore_index=True
)

display(asignaciones_zona_clima.head())

In [ ]:
# Construir una climatología por nodo, mes, día de semana y hora
renombres_clima = {
    'temperature_2m': 'TEMPERATURA_2M',
    'relative_humidity_2m': 'HUMEDAD_RELATIVA_2M',
    'apparent_temperature': 'SENSACION_TERMICA',
    'precipitation': 'PRECIPITACION',
    'rain': 'LLUVIA',
    'cloud_cover': 'NUBOSIDAD',
    'surface_pressure': 'PRESION_SUPERFICIE',
    'wind_speed_10m': 'VELOCIDAD_VIENTO_10M'
}
variables_clima_lineales = list(renombres_clima.values())
variables_clima = variables_clima_lineales + ['DIRECCION_VIENTO_10M']
columnas_clima_origen = list(renombres_clima) + ['wind_direction_10m']
columnas_cache = ['FECHA_HORA_CLIMA', *columnas_clima_origen]

climatologias_nodo = []
for archivo in asignaciones_zona_clima['ARCHIVO_CACHE'].drop_duplicates():
    clima = pd.read_csv(
        CARPETA_CACHE_CLIMA / archivo,
        usecols=columnas_cache,
        parse_dates=['FECHA_HORA_CLIMA']
    ).dropna(subset=columnas_clima_origen)

    if clima.empty:
        warnings.warn(f'El archivo {archivo} no contiene clima completo')
        continue

    clima['MES'] = clima['FECHA_HORA_CLIMA'].dt.month
    clima['DIA_SEMANA'] = clima['FECHA_HORA_CLIMA'].dt.dayofweek
    clima['HORA'] = clima['FECHA_HORA_CLIMA'].dt.hour
    direccion_radianes = np.deg2rad(clima['wind_direction_10m'])
    clima['_VIENTO_SIN'] = np.sin(direccion_radianes)
    clima['_VIENTO_COS'] = np.cos(direccion_radianes)

    agregaciones = {
        destino: (origen, 'mean')
        for origen, destino in renombres_clima.items()
    }
    agregaciones.update({
        '_VIENTO_SIN': ('_VIENTO_SIN', 'mean'),
        '_VIENTO_COS': ('_VIENTO_COS', 'mean'),
        '_OBSERVACIONES_CLIMA': ('FECHA_HORA_CLIMA', 'size')
    })

    climatologia = (
        clima.groupby(['MES', 'DIA_SEMANA', 'HORA'], as_index=False)
        .agg(**agregaciones)
    )
    climatologia['ARCHIVO_CACHE'] = archivo
    climatologias_nodo.append(climatologia)

climatologias_nodo = pd.concat(climatologias_nodo, ignore_index=True)
climatologia_zonas = asignaciones_zona_clima.merge(
    climatologias_nodo,
    on='ARCHIVO_CACHE',
    how='left',
    validate='m:m'
)

# Combinar ERA5 y ECMWF IFS ponderando por sus horas disponibles
variables_ponderadas = variables_clima_lineales + ['_VIENTO_SIN', '_VIENTO_COS']
for variable in variables_ponderadas:
    climatologia_zonas[f'_SUMA_{variable}'] = (
        climatologia_zonas[variable]
        * climatologia_zonas['_OBSERVACIONES_CLIMA']
    )

columnas_suma = [f'_SUMA_{variable}' for variable in variables_ponderadas]
climatologia_zonas = (
    climatologia_zonas.groupby(claves_patron, as_index=False)[
        [*columnas_suma, '_OBSERVACIONES_CLIMA']
    ]
    .sum()
)
for variable in variables_ponderadas:
    climatologia_zonas[variable] = (
        climatologia_zonas[f'_SUMA_{variable}']
        / climatologia_zonas['_OBSERVACIONES_CLIMA']
    )

climatologia_zonas['DIRECCION_VIENTO_10M'] = (
    np.degrees(np.arctan2(
        climatologia_zonas['_VIENTO_SIN'],
        climatologia_zonas['_VIENTO_COS']
    )) % 360
)
climatologia_zonas = climatologia_zonas[claves_patron + variables_clima]

dataset_ocurrencia = dataset_ocurrencia.merge(
    climatologia_zonas,
    on=claves_patron,
    how='left',
    validate='1:1'
)

### 4.4 Variables finales del modelo de ocurrencia

Las variables espacio-temporales del dataset final son `ZONA_CIUDAD`, `MES`, `DIA_SEMANA` y `HORA`. `LATITUD_CENTROIDE` y `LONGITUD_CENTROIDE` permiten representar o consultar cada zona geográficamente.

Las nueve variables meteorológicas son promedios históricos para el mismo patrón recurrente. No corresponden al clima de una fecha futura específica. `OCURRIO_ACCIDENTE` indica si la combinación apareció al menos una vez entre los accidentes históricos. Antes de construir el archivo final se valida que existan exactamente 302.400 combinaciones únicas y que ninguna fila quede sin clima ni coordenadas de centroide.

In [ ]:
# Validar la cuadrícula y la integración climática
filas_esperadas = NUMERO_ZONAS * 12 * 7 * 24
assert len(dataset_ocurrencia) == filas_esperadas
assert not dataset_ocurrencia.duplicated(claves_patron).any()
assert set(dataset_ocurrencia['OCURRIO_ACCIDENTE'].unique()) == {0, 1}
assert dataset_ocurrencia[variables_clima].notna().all().all()
assert dataset_ocurrencia[
    ['LATITUD_CENTROIDE', 'LONGITUD_CENTROIDE']
].notna().all().all()

print(f'Cuadrícula validada: {len(dataset_ocurrencia):,} filas')
display(dataset_ocurrencia.head())

## 5. Construcción de la tabla completa codificada de accidentes

Se construye una tabla consolidada con una sola fila por accidente, con el objetivo de realizar análisis descriptivos y explorar variables que posteriormente puedan ser útiles para estudiar la severidad de los eventos.

Las tablas `ACCIDENTE`, `CLIMA` y `VIA` se integran directamente a partir de sus llaves de relación. En el caso de `VEHICULO`, `ACTOR_VIAL` y `CAUSA`, un mismo accidente puede estar relacionado con varios registros. Por esta razón, estas tablas se resumen previamente antes de realizar la integración, evitando así la duplicación de accidentes en la tabla final.

> **Nota metodológica:** esta tabla incluye variables que solo se conocen después de que el accidente ha ocurrido, como las causas, los vehículos involucrados o las características de los actores viales. Por esta razón, no debe utilizarse de forma completa para entrenar el modelo de ocurrencia de accidentes. Su uso está orientado principalmente al análisis descriptivo y a posibles análisis posteriores de severidad.

### 5.1 Resumen de vehículos

Para cada accidente se calcula la cantidad total de vehículos involucrados. Adicionalmente, se generan indicadores que permiten identificar los principales tipos de vehículos presentes en cada evento.

Este resumen permite integrar la información de la tabla `VEHICULO` sin generar múltiples filas para un mismo accidente.

In [ ]:
# Caracterización de vehículos involucrados en los accidentes
vehiculo_cod = TABLAS_CODIFICADAS['VEHICULO']

def codigos_con_texto(tabla, columna, texto):
    mapa = MAPAS_CATEGORIAS[(tabla, columna)]

    return {
        codigo
        for categoria, codigo in mapa.items()
        if texto.upper() in str(categoria).upper()
    }

vehiculos_aux = vehiculo_cod.assign(
    TIENE_MOTOCICLETA=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'MOTO')).astype(int),

    TIENE_BICICLETA=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'BICI')).astype(int),

    TIENE_AUTOMOVIL=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'AUTOMOVIL')).astype(int),

    TIENE_BUS=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'BUS')).astype(int),

    TIENE_CAMION=vehiculo_cod['CLASE']
        .isin(codigos_con_texto('VEHICULO', 'CLASE', 'CAMION')).astype(int)
)

resumen_vehiculos = (
    vehiculos_aux
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_VEHICULOS=('VEHICULO_ID', 'size'),
        CLASES_VEHICULO_DISTINTAS=('CLASE', 'nunique'),
        SERVICIOS_VEHICULO_DISTINTOS=('SERVICIO', 'nunique'),
        TIENE_MOTOCICLETA=('TIENE_MOTOCICLETA', 'max'),
        TIENE_BICICLETA=('TIENE_BICICLETA', 'max'),
        TIENE_AUTOMOVIL=('TIENE_AUTOMOVIL', 'max'),
        TIENE_BUS=('TIENE_BUS', 'max'),
        TIENE_CAMION=('TIENE_CAMION', 'max')
    )
)

display(resumen_vehiculos.head())

### 5.2 Resumen de actores y causas

La información de los actores viales se resume para cada accidente mediante la cantidad total de personas involucradas, variables relacionadas con la edad e indicadores que permiten identificar su condición y estado dentro del evento.

Por su parte, las causas se resumen a partir de la cantidad de registros asociados a cada accidente y la diversidad de causas identificadas por su nombre.

De esta manera, ambas fuentes pueden integrarse a la tabla consolidada manteniendo una sola fila por accidente y evitando la duplicación de registros.

In [ ]:
# Caracterización de actores y causas de los accidentes
actor_cod = TABLAS_CODIFICADAS['ACTOR_VIAL']
causa_cod = TABLAS_CODIFICADAS['CAUSA']

# Resumen de actores por accidente
actor_aux = actor_cod.assign(
    ES_CONDUCTOR=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'CONDUCTOR')).astype(int),

    ES_PASAJERO=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'PASAJ')).astype(int),

    ES_PEATON=actor_cod['CONDICION']
        .isin(codigos_con_texto('ACTOR_VIAL', 'CONDICION', 'PEATON')).astype(int),

    ES_HERIDO=actor_cod['ESTADO']
        .isin(codigos_con_texto('ACTOR_VIAL', 'ESTADO', 'HERID')).astype(int),

    ES_MUERTO=actor_cod['ESTADO']
        .isin(codigos_con_texto('ACTOR_VIAL', 'ESTADO', 'MUERT')).astype(int)
)

resumen_actores = (
    actor_aux
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_ACTORES=('ACTOR_ID', 'size'),
        EDAD_PROMEDIO=('EDAD', 'mean'),
        EDAD_MINIMA=('EDAD', 'min'),
        EDAD_MAXIMA=('EDAD', 'max'),
        CANTIDAD_CONDUCTORES=('ES_CONDUCTOR', 'sum'),
        CANTIDAD_PASAJEROS=('ES_PASAJERO', 'sum'),
        CANTIDAD_PEATONES=('ES_PEATON', 'sum'),
        CANTIDAD_HERIDOS=('ES_HERIDO', 'sum'),
        CANTIDAD_MUERTOS=('ES_MUERTO', 'sum')
    )
)

# Resumen de causas por accidente
resumen_causas = (
    causa_cod
    .groupby('ACCIDENTE_ID', as_index=False)
    .agg(
        CANTIDAD_CAUSAS=('CAUSA_ID', 'size'),
        CAUSAS_DISTINTAS=('NOMBRE', 'nunique'),
        TIPOS_CAUSA_DISTINTOS=('TIPO', 'nunique')
    )
)

display(resumen_actores.head())
display(resumen_causas.head())

### 5.3 Integración de la tabla completa

Finalmente, se integran las seis fuentes de información para construir una tabla consolidada con una sola fila por cada `ACCIDENTE_ID`.

Durante esta integración, los valores faltantes asociados a variables categóricas codificadas, indicadores y conteos se completan con `0`, manteniendo así una estructura consistente para el análisis posterior.

El resultado corresponde a una tabla completa por accidente, que reúne en un mismo registro la información disponible sobre el evento, el clima, la vía, los vehículos, los actores viales y las causas asociadas.


In [ ]:
# Integrar una fila por accidente
accidente_cod = TABLAS_CODIFICADAS['ACCIDENTE']
clima_cod = TABLAS_CODIFICADAS['CLIMA']
via_cod = TABLAS_CODIFICADAS['VIA']

TABLA_COMPLETA_ACCIDENTES = (
    accidente_cod
    .merge(clima_cod, on='CLIMA_ID', how='left', validate='m:1')
    .merge(via_cod, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_vehiculos, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_actores, on='ACCIDENTE_ID', how='left', validate='1:1')
    .merge(resumen_causas, on='ACCIDENTE_ID', how='left', validate='1:1')
)

# Completar categorías sin información
columnas_categoricas = [
    'CLASE_ACCIDENTE',
    'GEOMETRIA_PLANTA',
    'GEOMETRIA_TERRENO',
    'GEOMETRIA_SECCION',
    'SENTIDO_VIA',
    'SUPERFICIE_RODADURA',
    'ESTADO_VIA',
    'CONDICION_VIA',
    'ILUMINACION_ARTIFICIAL',
    'SEMAFORO'
]

TABLA_COMPLETA_ACCIDENTES[columnas_categoricas] = (
    TABLA_COMPLETA_ACCIDENTES[columnas_categoricas]
    .fillna(0)
    .astype(int)
)

# Completar variables de conteo
columnas_conteo = [
    'CANTIDAD_VEHICULOS',
    'CLASES_VEHICULO_DISTINTAS',
    'SERVICIOS_VEHICULO_DISTINTOS',
    'TIENE_MOTOCICLETA',
    'TIENE_BICICLETA',
    'TIENE_AUTOMOVIL',
    'TIENE_BUS',
    'TIENE_CAMION',
    'CANTIDAD_ACTORES',
    'CANTIDAD_CONDUCTORES',
    'CANTIDAD_PASAJEROS',
    'CANTIDAD_PEATONES',
    'CANTIDAD_HERIDOS',
    'CANTIDAD_MUERTOS',
    'CANTIDAD_CAUSAS',
    'CAUSAS_DISTINTAS',
    'TIPOS_CAUSA_DISTINTOS'
]

TABLA_COMPLETA_ACCIDENTES[columnas_conteo] = (
    TABLA_COMPLETA_ACCIDENTES[columnas_conteo]
    .fillna(0)
    .astype(int)
)

display(TABLA_COMPLETA_ACCIDENTES.head())

## 6. Construcción de los datasets finales para modelado

A partir de las tablas procesadas, se construyen **dos datasets de modelado**, cada uno orientado a un objetivo analítico diferente, y se conserva una **tercera tabla consolidada** para análisis histórico y exploratorio.

La lógica analítica de **PrediRuta** se estructura alrededor de tres preguntas principales:

1. **Ocurrencia:** ¿qué patrones recurrentes de zona, mes, día de la semana y hora han registrado al menos un siniestro en el histórico?

2. **Tipo de evento:** dado que ocurre un siniestro, ¿qué clase de accidente es más probable que se presente?

3. **Severidad:** dado que ocurre un siniestro, ¿qué probabilidad existe de que corresponda a un evento grave?

Las **variables climáticas** se incorporan como un grupo adicional de predictores. Para el modelo de ocurrencia, estas variables corresponden a condiciones climáticas históricas asociadas al patrón **zona–mes–día–hora**. Para los modelos basados en eventos, corresponden a las condiciones climáticas registradas o estimadas para el momento y ubicación de cada accidente.

Esta estructura permite evaluar el aporte del clima mediante la comparación de cada modelo bajo dos escenarios: **sin variables climáticas** y **con variables climáticas**.

> **Nota metodológica:** las variables que únicamente se conocen después de ocurrido un accidente —como los vehículos involucrados, los actores viales, el número de heridos o fallecidos y las causas registradas— no se utilizan como predictores de la ocurrencia de eventos futuros. Estas variables se conservan en la tabla consolidada para fines de caracterización, análisis histórico y exploración de posibles relaciones con el tipo y la severidad de los siniestros.


### 6.1 Dataset de ocurrencia

### Dataset de ocurrencia

El primer dataset tiene como unidad de análisis una combinación recurrente **zona–mes–día de la semana–hora**. Para su construcción se genera la cuadrícula completa de las **150 zonas** definidas para el análisis y todas las combinaciones posibles de mes, día de la semana y hora.

La variable objetivo es `OCURRIO_ACCIDENTE`, definida de la siguiente manera:

* `1`: se registró al menos un accidente histórico para la combinación **zona–mes–día de la semana–hora**.
* `0`: no se registraron accidentes históricos para dicha combinación.

Cada fila incluye las **coordenadas del centroide de la zona**, las variables temporales correspondientes y las **condiciones climáticas históricas promedio** asociadas al mismo patrón de mes, día de la semana y hora. De esta forma, las variables climáticas están disponibles tanto para las observaciones con ocurrencia (`1`) como para aquellas sin ocurrencia (`0`).

Para evaluar el aporte de la información meteorológica se plantean dos escenarios de modelado:

* **Modelo base:** utiliza únicamente variables espaciales y temporales.
* **Modelo con clima:** utiliza las mismas variables espaciales y temporales e incorpora adicionalmente las variables meteorológicas.

La comparación entre ambos escenarios debe realizarse sobre **exactamente las mismas observaciones y la misma partición de entrenamiento y prueba**, de manera que cualquier diferencia en desempeño pueda atribuirse a la incorporación de las variables climáticas y no a cambios en los datos utilizados.

Es importante señalar que la salida de este modelo representa la **asociación de una combinación espacio-temporal con patrones históricos de ocurrencia de siniestros**. Por tanto, no debe interpretarse directamente como una probabilidad calibrada de que ocurra un accidente en una fecha futura específica, sino como una estimación del patrón histórico recurrente de accidentalidad para determinadas condiciones espaciales y temporales.


In [ ]:
# Definir grupos de predictores para el modelo de ocurrencia
PREDICTORES_ESPACIOTEMPORALES = [
    'MES',
    'DIA_SEMANA',
    'HORA',
    'ZONA_CIUDAD'
]

PREDICTORES_CLIMA = variables_clima.copy()

# Construir dataset final de ocurrencia
columnas_ocurrencia = [
    *PREDICTORES_ESPACIOTEMPORALES,
    'LATITUD_CENTROIDE',
    'LONGITUD_CENTROIDE',
    *PREDICTORES_CLIMA,
    'OCURRIO_ACCIDENTE'
]

DATASET_OCURRENCIA = (
    dataset_ocurrencia[columnas_ocurrencia]
    .sort_values(['ZONA_CIUDAD', 'MES', 'DIA_SEMANA', 'HORA'])
    .reset_index(drop=True)
)

display(DATASET_OCURRENCIA)

print(f'Casos con accidente: '
      f'{DATASET_OCURRENCIA["OCURRIO_ACCIDENTE"].eq(1).sum():,}')
print(f'Combinaciones sin accidente histórico: '
      f'{DATASET_OCURRENCIA["OCURRIO_ACCIDENTE"].eq(0).sum():,}')

### 6.2 Dataset de eventos ocurridos

El segundo dataset contiene únicamente accidentes que efectivamente ocurrieron y se utilizará para desarrollar modelos complementarios condicionados a la existencia de un evento.

A partir de este conjunto se podrán analizar dos variables objetivo:

* `CLASE_ACCIDENTE`: permite estimar qué tipo de accidente podría presentarse una vez ocurre un evento.
* `OBJETIVO_GRAVE`: permite estimar la probabilidad de que el accidente corresponda a un evento grave.

In [ ]:
# Construir dataset de accidentes ocurridos directamente desde ACCIDENTE + CLIMA
eventos = (
    TABLAS_CODIFICADAS['ACCIDENTE']
    .merge(
        TABLAS_CODIFICADAS['CLIMA'],
        on='CLIMA_ID',
        how='left',
        validate='m:1'
    )
    .copy()
)

# Asignar cada accidente a una celda de 500 m para los modelos de evento
x_evento, y_evento = wgs84_a_bogota.transform(
    eventos['LONGITUD'].to_numpy(),
    eventos['LATITUD'].to_numpy()
)

eventos['CELDA_X'] = (
    np.floor(np.asarray(x_evento) / TAMANO_CELDA_METROS)
    .astype('int32')
)

eventos['CELDA_Y'] = (
    np.floor(np.asarray(y_evento) / TAMANO_CELDA_METROS)
    .astype('int32')
)

eventos['CELDA_ID'] = (
    eventos['CELDA_X'].astype(str)
    + '_'
    + eventos['CELDA_Y'].astype(str)
)

# Derivar variables temporales
eventos['ANIO'] = eventos['FECHA_HORA'].dt.year
eventos['MES'] = eventos['FECHA_HORA'].dt.month
eventos['DIA_SEMANA'] = eventos['FECHA_HORA'].dt.dayofweek
eventos['HORA'] = eventos['FECHA_HORA'].dt.hour
eventos['FIN_SEMANA'] = (
    eventos['DIA_SEMANA'].isin([5, 6]).astype(int)
)

# Predictores disponibles antes o durante una consulta futura
PREDICTORES_EVENTO = [
    'CELDA_X',
    'CELDA_Y',
    'ANIO',
    'MES',
    'DIA_SEMANA',
    'HORA',
    'FIN_SEMANA'
]

columnas_eventos = [
    'ACCIDENTE_ID',
    'CELDA_ID',
    'FECHA_HORA',
    'LATITUD',
    'LONGITUD',
    *PREDICTORES_EVENTO,
    *PREDICTORES_CLIMA,
    'CLASE_ACCIDENTE',
    'OBJETIVO_GRAVE'
]

DATASET_EVENTOS = (
    eventos[columnas_eventos]
    .sort_values('FECHA_HORA')
    .reset_index(drop=True)
)

display(DATASET_EVENTOS)

### 6.3 Estructura de experimentación

Los datasets construidos permiten plantear tres objetivos predictivos diferentes sin duplicar innecesariamente la información.

| Componente     | Dataset              | Variable objetivo   | Comparación propuesta                     |
| -------------- | -------------------- | ------------------- | ----------------------------------------- |
| Ocurrencia     | `DATASET_OCURRENCIA` | `OCURRIO_ACCIDENTE` | Espacio-tiempo vs. espacio-tiempo + clima |
| Tipo de evento | `DATASET_EVENTOS`    | `CLASE_ACCIDENTE`   | Espacio-tiempo vs. espacio-tiempo + clima |
| Severidad      | `DATASET_EVENTOS`    | `OBJETIVO_GRAVE`    | Espacio-tiempo vs. espacio-tiempo + clima |

Las variables meteorológicas se manejan como un **bloque adicional de predictores**. Para cada uno de los tres objetivos se plantea entrenar primero un modelo base utilizando únicamente variables espaciales y temporales y, posteriormente, repetir el experimento incorporando las variables climáticas.


In [ ]:
# Resumen de los datasets disponibles para la etapa de modelado
resumen_modelado = pd.DataFrame([
    {
        'DATASET': 'DATASET_OCURRENCIA',
        'UNIDAD_ANALISIS': 'Zona-mes-día-hora',
        'FILAS': len(DATASET_OCURRENCIA),
        'COLUMNAS': len(DATASET_OCURRENCIA.columns),
        'OBJETIVO': 'OCURRIO_ACCIDENTE'
    },
    {
        'DATASET': 'DATASET_EVENTOS',
        'UNIDAD_ANALISIS': 'Accidente',
        'FILAS': len(DATASET_EVENTOS),
        'COLUMNAS': len(DATASET_EVENTOS.columns),
        'OBJETIVO': 'CLASE_ACCIDENTE / OBJETIVO_GRAVE'
    },
    {
        'DATASET': 'TABLA_COMPLETA_ACCIDENTES',
        'UNIDAD_ANALISIS': 'Accidente',
        'FILAS': len(TABLA_COMPLETA_ACCIDENTES),
        'COLUMNAS': len(TABLA_COMPLETA_ACCIDENTES.columns),
        'OBJETIVO': 'Análisis histórico y exploratorio'
    }
])

display(resumen_modelado)

### 6.4 Tabla completa de accidentes

`TABLA_COMPLETA_ACCIDENTES` se conserva como la tabla con mayor nivel de detalle de los eventos. Contiene una fila por accidente e integra la información disponible de las fuentes de accidente, clima, vía, vehículos, actores viales y causas.

In [ ]:
# Verificar la estructura de la tabla completa de accidentes
display(TABLA_COMPLETA_ACCIDENTES)


## 7. Exportación

Como resultado del procesamiento se exportan cuatro productos finales:

1. `DICCIONARIO_CATEGORIAS.csv`: contiene la codificación utilizada para transformar las variables categóricas y permite interpretar posteriormente los códigos asignados.

2. `DATASET_OCURRENCIA.csv`: contiene la cuadrícula de patrones recurrentes zona-mes-día-hora, su etiqueta de ocurrencia histórica y la climatología correspondiente.

3. `DATASET_EVENTOS.csv`: contiene únicamente accidentes ocurridos y se utiliza para modelar el tipo de evento y la severidad, condicionados a que exista un siniestro.

4. `TABLA_COMPLETA_ACCIDENTES.csv`: reúne la información consolidada por accidente y se conserva para análisis histórico, exploración de relaciones y evaluación posterior de posibles variables.

Los modelos con y sin clima se construirán posteriormente a partir de estos mismos datasets, modificando únicamente los grupos de predictores utilizados en cada experimento. De esta manera, será posible comparar de forma consistente el aporte adicional de las variables meteorológicas.


In [ ]:
# Ruta adicional para el dataset de eventos
RUTA_EVENTOS = CARPETA_SALIDA / 'DATASET_EVENTOS.csv'

# Exportar productos finales
DICCIONARIO_CATEGORIAS.to_csv(
    RUTA_DICCIONARIO,
    index=False,
    encoding='utf-8-sig'
)

DATASET_OCURRENCIA.to_csv(
    RUTA_OCURRENCIA,
    index=False,
    encoding='utf-8-sig'
)

DATASET_EVENTOS.to_csv(
    RUTA_EVENTOS,
    index=False,
    encoding='utf-8-sig'
)

TABLA_COMPLETA_ACCIDENTES.to_csv(
    RUTA_COMPLETA,
    index=False,
    encoding='utf-8-sig'
)

print('Archivos generados:')
print(f'- Diccionario:    {DICCIONARIO_CATEGORIAS.shape}')
print(f'- Ocurrencia:     {DATASET_OCURRENCIA.shape}')
print(f'- Eventos:        {DATASET_EVENTOS.shape}')
print(f'- Tabla completa: {TABLA_COMPLETA_ACCIDENTES.shape}')